# 22 — Barrido fino de timestamps: ¿hay más señal en los segundos?

**Motivación.** `duracion_dias` (`stop_time` − `start_time`) sumó +0,0044 al modelo final porque la señal estaba en los **segundos**: las publicaciones que duran 60 días + unos segundos son casi todas `used` (`18_barrido_campos`). En `06_eda_nuevas` y en la 18, las demás fechas se miraron en **días**. Acá se repite el barrido sobre los cuatro campos de fecha (`start_time`, `stop_time`, `date_created`, `last_updated`) a nivel de segundos y milisegundos.

**Plan.**
1. Diferencias entre los 6 pares de fechas, en segundos exactos, y su **sobrante** respecto de un número entero de días.
2. Componentes de cada fecha: segundo, minuto, hora, día de la semana, día del mes.
3. Milisegundos y valores "redondos" (segundos = 0, milisegundos ≠ 0): posibles huellas de cargas automáticas.
4. Coincidencias exactas entre fechas, como flags.
5. Cargas masivas por vendedor: tiempo hasta la publicación anterior/siguiente del mismo vendedor y cuántas publicó en la misma hora y el mismo día (train + test juntos, sin target).
6. Medición, igual que en la 18: **señal sola** y **aporte marginal** sobre el HistGB P7a con duración (comparación pareada fold por fold). Lo que pase se confirma en el promedio final (CatBoost + HistGB, 0,9420) con las semillas 42 y 7.

**No es leakage (partes 1–4):** todas las features son funciones **determinísticas** de las fechas de cada fila. No usan `condition` y se calculan igual en train y en test, así que pueden ir afuera del `Pipeline` (como `duracion_dias`). La parte 5 se justifica aparte.

**Material.** Generar variables nuevas a partir de las existentes: sección "Feature Engineering" de `material/Variables Categóricas, Datos Faltantes y Feature Engineering/Copia de AP_Categóricas.ipynb`. Gradient Boosting: `material/Arboles/Árboles.pdf`. El material no trae nada específico sobre fechas.

**Por qué los segundos pueden "perderse" en el modelo actual** (no está en el material, es de la documentación de sklearn): `HistGradientBoostingClassifier` agrupa cada numérica en a lo sumo **255 cajas** (*bins*) por cuantiles antes de buscar cortes. `gap_actualizacion` tiene decenas de miles de valores distintos, así que un patrón del tipo "termina en 7 segundos" queda mezclado dentro de cajas de horas. Una feature aparte con solo el sobrante de segundos sí lo hace visible.

In [1]:
import sys
sys.path.append("..")
import os, json, time, warnings, itertools
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
SEED = 42


def leer_fechas(ruta):
    # Lee las 4 fechas y el vendedor tal como vienen en el jsonlines (sin que pandas las convierta)
    filas = []
    with open(ruta) as f:
        for linea in f:
            d = json.loads(linea)
            filas.append({c: d[c] for c in ["seller_id", "start_time", "stop_time", "date_created", "last_updated"]})
    return pd.DataFrame(filas)


# Train (70.000 filas) y test (30.000) juntos, en ese orden. Solo fechas y vendedor: nada del target.
fechas = pd.concat([leer_fechas("../data/train_data.jsonlines").assign(es_test=False),
                    leer_fechas("../data/test_data.jsonlines").assign(es_test=True)], ignore_index=True)
es_test = fechas["es_test"].to_numpy()

# Las 4 fechas como milisegundos desde 1970 (UTC), en enteros: así las diferencias son exactas.
# start_time y stop_time ya vienen así; date_created y last_updated vienen como texto ISO ("...T19:19:14.000Z").
ms = pd.DataFrame({
    "inicio": fechas["start_time"].astype("int64"),
    "fin": fechas["stop_time"].astype("int64"),
    "creada": pd.to_datetime(fechas["date_created"], utc=True).dt.as_unit("ms").astype("int64"),
    "actualizada": pd.to_datetime(fechas["last_updated"], utc=True).dt.as_unit("ms").astype("int64"),
})

# Target solo de train (para describir; las features no lo usan)
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
assert (crudo["seller_id"].to_numpy() == fechas.loc[~es_test, "seller_id"].to_numpy()).all()  # mismo orden de filas

print(fechas.head(3)[["start_time", "stop_time", "date_created", "last_updated"]])
print(ms.head(3))
print("Nulos:", ms.isna().sum().sum(), "| % new en train:", round(y.mean(), 4))

      start_time      stop_time              date_created              last_updated
0  1443813553000  1448997553000  2015-10-02T19:19:14.000Z  2015-10-02T19:19:16.000Z
1  1444247890000  1449431890000  2015-10-07T19:58:10.000Z  2015-10-07T19:58:12.000Z
2  1442537290000  1447721290000  2015-09-18T00:48:10.000Z  2015-10-05T21:35:39.000Z
          inicio            fin         creada    actualizada
0  1443813553000  1448997553000  1443813554000  1443813556000
1  1444247890000  1449431890000  1444247890000  1444247892000
2  1442537290000  1447721290000  1442537290000  1444080939000
Nulos: 0 | % new en train: 0.537


In [2]:
def tabla(serie, top=8, indice=None):
    # Valores más frecuentes de una serie (definida en train + test): filas de train, % de train, % de test y % de new
    tr, te = serie[~es_test], serie[es_test]
    conteo = tr.value_counts()
    indice = conteo.index[:top] if indice is None else indice
    return pd.DataFrame({
        "filas_train": conteo.reindex(indice).fillna(0).astype(int),
        "pct_train": tr.value_counts(normalize=True).reindex(indice).fillna(0),
        "pct_test": te.value_counts(normalize=True).reindex(indice).fillna(0),
        "pct_new": y.groupby(tr.to_numpy()).mean().reindex(indice),
    }).round(4)


def resumen_flags(flags):
    # Para cada flag 0/1: % de filas con 1 en train y en test, y % de new con 1 y con 0
    filas = []
    for c in flags:
        tr = flags.loc[~es_test, c].to_numpy()
        filas.append({"flag": c, "filas_train_1": int(tr.sum()), "pct_train_1": tr.mean(),
                      "pct_test_1": flags.loc[es_test, c].mean(),
                      "pct_new_si_1": y[tr == 1].mean() if tr.sum() else np.nan, "pct_new_si_0": y[tr == 0].mean()})
    return pd.DataFrame(filas).set_index("flag").round(4)

## 1. Diferencias entre pares de fechas, en segundos exactos

Para cada uno de los 6 pares (orden cronológico típico: creada → inicio → actualizada → fin) se calcula `b − a` en segundos (con milisegundos como decimales). Además, el **sobrante respecto de días redondos**: la diferencia menos el número entero de días más cercano, en segundos. Ejemplo: 60 días + 7 s → sobrante = +7; 59 días, 23 h, 59 min, 58 s → sobrante = −2. Es exactamente lo que hizo visible la señal de la duración.

Por par se muestran las diferencias exactas más frecuentes y los sobrantes más frecuentes, con su % de filas en train y en test y el % de `new` (el promedio general es 53,7%).

In [3]:
DIA = 86400
pares = list(itertools.combinations(["creada", "inicio", "actualizada", "fin"], 2))
dif = pd.DataFrame({f"{a}_a_{b}": (ms[b] - ms[a]) / 1000 for a, b in pares})  # segundos exactos, b − a
sobrante = dif - (dif / DIA).round() * DIA                                     # segundos respecto del día entero más cercano

pd.DataFrame({
    "valores_distintos": dif[~es_test].nunique(),
    "pct_negativas": (dif[~es_test] < 0).mean(),
    "mediana_dias": dif[~es_test].median() / DIA,
    "sobrantes_distintos": sobrante[~es_test].nunique(),
    "pct_sobrante_cero": (sobrante[~es_test] == 0).mean(),
    "pct_sobrante_cero_test": (sobrante[es_test] == 0).mean(),
}).round(4)

,valores_distintos,pct_negativas,mediana_dias,sobrantes_distintos,pct_sobrante_cero,pct_sobrante_cero_test
creada_a_inicio,322,0.1980,0.0000,315,0.7971,0.7994
creada_a_actualizada,24042,0.0002,0.0001,21474,0.0211,0.0214
creada_a_fin,2079,0.0000,60.0000,2007,0.7658,0.7667
inicio_a_actualizada,23938,0.0000,0.0001,21392,0.0132,0.0128
inicio_a_fin,1793,0.0000,60.0000,1759,0.9563,0.9534
actualizada_a_fin,23924,0.0000,59.9999,21100,0.0132,0.0128


In [4]:
for par in dif:
    print(f"=== {par} ===")
    print("Diferencias exactas más frecuentes (segundos):")
    print(tabla(dif[par]).to_string())
    print("Sobrantes más frecuentes (segundos respecto del día entero más cercano):")
    print(tabla(sobrante[par]).to_string(), end="\n\n")

=== creada_a_inicio ===
Diferencias exactas más frecuentes (segundos):
                 filas_train  pct_train  pct_test  pct_new
creada_a_inicio                                           
 0.0                   55789     0.7970    0.7993   0.5352
-1.0                   13622     0.1946    0.1929   0.5493
-2.0                     171     0.0024    0.0025   0.5789
-3.0                      41     0.0006    0.0008   0.7561
-4.0                      21     0.0003    0.0002   0.5714
-5.0                       6     0.0001    0.0001   0.5000
 252.0                     4     0.0001    0.0000   0.2500
 262.0                     3     0.0000    0.0000   0.6667
Sobrantes más frecuentes (segundos respecto del día entero más cercano):
                 filas_train  pct_train  pct_test  pct_new
creada_a_inicio                                           
 0.0                   55794     0.7971    0.7994   0.5352
-1.0                   13625     0.1946    0.1929   0.5492
-2.0                     171  

**Lectura de la parte 1.**
- **Lo único que separa claramente es la duración, otra vez.** Los sobrantes de +5 a +11 s en `inicio_a_fin` (y los mismos en `creada_a_fin`) tienen 0-2% de `new`: son las publicaciones de "60 días + unos segundos" que ya captura `duracion_dias`.
- `creada_a_inicio` vale 0 s (80%) o −1 s (19%): `start_time` queda en el mismo segundo que `date_created` o uno antes. Casi sin diferencia de `new` (53,5% contra 54,9%). Las pocas filas con otras diferencias (~0,8%) tienen 42% de `new`.
- `creada_a_actualizada`, `inicio_a_actualizada` y `actualizada_a_fin`: la mayoría se actualiza 1-7 segundos después de crearse, con % de `new` entre 44% y 53% según el segundo exacto. Sin una huella fuerte.
- Train y test tienen las mismas proporciones en todos los valores (el test es una muestra del mismo universo).

## 2. Componentes de cada fecha

Segundo, minuto, hora, día de la semana (0 = lunes) y día del mes de cada fecha, en **hora de Argentina** (UTC−3; Argentina no tuvo horario de verano entre 2013 y 2015). Para cada componente se resume cuánto varía el % de `new` entre sus valores (solo valores con ≥ 300 filas en train, para no leer ruido). Si un componente no tuviera relación con la condición, todos sus valores darían ~53,7%.

Ojo con la variabilidad esperable por azar: con 300 filas, el desvío de un porcentaje es √(0,54·0,46/300) ≈ 2,9 puntos, así que diferencias de ±5 puntos entre muchos valores son normales. Con 70.000/60 ≈ 1.170 filas por segundo, el azar da ±1,5 puntos.

In [5]:
local = {c: pd.to_datetime(ms[c], unit="ms") - pd.Timedelta(hours=3) for c in ms}   # hora de Argentina
componentes = pd.DataFrame({f"{c}_{nombre}": f(t) for c, t in local.items() for nombre, f in [
    ("segundo", lambda t: t.dt.second), ("minuto", lambda t: t.dt.minute), ("hora", lambda t: t.dt.hour),
    ("dia_semana", lambda t: t.dt.dayofweek), ("dia_mes", lambda t: t.dt.day)]})


def dispersion(serie, min_filas=300):
    g = y.groupby(serie[~es_test].to_numpy())
    n, p = g.size(), g.mean()
    p = p[n >= min_filas]
    return {"valores": len(n), "valores_300+": len(p), "pct_new_min": p.min(), "valor_min": p.idxmin(),
            "pct_new_max": p.max(), "valor_max": p.idxmax(), "rango": p.max() - p.min()}


resumen_comp = pd.DataFrame({c: dispersion(componentes[c]) for c in componentes}).T
resumen_comp.sort_values("rango", ascending=False)

,valores,valores_300+,pct_new_min,valor_min,pct_new_max,valor_max,rango
actualizada_hora,24.0,23.0,0.395904,6.0,0.609971,4.0,0.214066
creada_hora,24.0,23.0,0.432735,6.0,0.613115,4.0,0.180379
inicio_hora,24.0,23.0,0.434685,6.0,0.613115,4.0,0.178430
actualizada_dia_mes,31.0,31.0,0.441421,20.0,0.618743,14.0,0.177322
fin_hora,24.0,23.0,0.436073,6.0,0.602501,9.0,0.166428
actualizada_dia_semana,7.0,7.0,0.462995,6.0,0.566539,2.0,0.103544
fin_segundo,60.0,60.0,0.473459,5.0,0.575503,30.0,0.102044
fin_dia_mes,31.0,31.0,0.473191,19.0,0.574555,24.0,0.101364
inicio_segundo,60.0,60.0,0.474174,5.0,0.571206,24.0,0.097032
creada_segundo,60.0,60.0,0.485420,5.0,0.580218,24.0,0.094798


In [6]:
# Detalle de los componentes con más rango: % de new por valor
for c in resumen_comp.sort_values("rango", ascending=False).index[:4]:
    print(f"=== {c} ===")
    t = tabla(componentes[c], indice=sorted(componentes[c].unique()))
    print(t.T.to_string(), end="\n\n")

=== actualizada_hora ===
                    0          1         2         3         4         5         6          7          8          9          10         11         12         13         14         15         16         17         18         19         20         21         22         23
filas_train  2031.0000  1298.0000  729.0000  447.0000  341.0000  287.0000  586.0000  1042.0000  2639.0000  3804.0000  4950.0000  4803.0000  4639.0000  4153.0000  3986.0000  4232.0000  4328.0000  4246.0000  4565.0000  4196.0000  3753.0000  3290.0000  2899.0000  2756.0000
pct_train       0.0290     0.0185    0.0104    0.0064    0.0049    0.0041    0.0084     0.0149     0.0377     0.0543     0.0707     0.0686     0.0663     0.0593     0.0569     0.0605     0.0618     0.0607     0.0652     0.0599     0.0536     0.0470     0.0414     0.0394
pct_test        0.0294     0.0185    0.0106    0.0067    0.0049    0.0042    0.0080     0.0147     0.0400     0.0543     0.0690     0.0684     0.0686     0.0615  

**Lectura de la parte 2.**
- **Hora:** el % de `new` va de ~40% (6 de la mañana) a ~61% (4 de la mañana), con un patrón diurno suave (más `new` en horario comercial, 8-17 h). Ya está en el modelo como `hora_inicio`.
- **Día del mes de `last_updated`:** de 44% a 62%. Probablemente es un efecto de calendario (en qué momento del período de extracción se actualizó), muy ligado a `gap_actualizacion`, que ya está en el modelo.
- **Segundo y minuto:** rangos de 7-10 puntos entre 60 valores de ~1.170 filas cada uno. El azar solo da ±1,5 puntos por valor, así que el máximo y el mínimo de 60 valores pueden separarse ~6-7 puntos sin que haya nada. No hay un valor que se destaque claramente.
- Ningún componente muestra una huella como la de la duración (un valor con casi 0% o 100% de `new`).

## 3. Milisegundos y valores redondos

Una carga automática (tiendas que publican por API) o un proceso interno de la plataforma puede dejar una huella en los valores: milisegundos distintos de 000, segundos exactamente en 0, horas en punto. Por azar se esperaría segundo = 0 en 1/60 ≈ 1,67% de las filas y hh:00:00 en 1/3.600 ≈ 0,03%.

In [7]:
seg = ms // 1000   # segundos desde 1970
redondos = pd.DataFrame({f"{c}_{nombre}": f(c).astype(int) for c in ms for nombre, f in [
    ("ms_no_cero", lambda c: ms[c] % 1000 != 0),
    ("segundo_cero", lambda c: seg[c] % 60 == 0),
    ("hora_en_punto", lambda c: seg[c] % 3600 == 0)]})
resumen_flags(redondos)

,filas_train_1,pct_train_1,pct_test_1,pct_new_si_1,pct_new_si_0
flag,,,,,
inicio_ms_no_cero,0,0.0000,0.0000,NaN,0.5370
inicio_segundo_cero,1270,0.0181,0.0179,0.5268,0.5372
inicio_hora_en_punto,87,0.0012,0.0012,0.2644,0.5373
fin_ms_no_cero,96,0.0014,0.0013,0.3021,0.5373
fin_segundo_cero,1292,0.0185,0.0180,0.5325,0.5371
fin_hora_en_punto,84,0.0012,0.0012,0.2619,0.5373
creada_ms_no_cero,0,0.0000,0.0000,NaN,0.5370
creada_segundo_cero,1148,0.0164,0.0159,0.5366,0.5370
creada_hora_en_punto,20,0.0003,0.0003,0.5500,0.5370


In [8]:
# ¿Qué milisegundos aparecen cuando no son 000?
for c in ["fin", "actualizada"]:
    print(f"=== {c}: milisegundos (solo filas con ms ≠ 0) ===")
    milis = (ms[c] % 1000).where(ms[c] % 1000 != 0)
    print(tabla(milis.dropna().astype(int).reindex(ms.index)).to_string(), end="\n\n")

# Las filas de fin con ms ≠ 0: ¿qué duración tienen?
filas_fin = (ms["fin"] % 1000 != 0) & ~es_test
print("Duración (días, 5 decimales) de las filas de train con ms de fin ≠ 0:")
print((dif.loc[filas_fin, "inicio_a_fin"] / DIA).round(5).value_counts().head(8))

=== fin: milisegundos (solo filas con ms ≠ 0) ===
       filas_train  pct_train  pct_test  pct_new
fin                                             
558.0            5     0.0521    0.0000      0.0
816.0            3     0.0312    0.0000      0.0
518.0            3     0.0312    0.0263      0.0
475.0            3     0.0312    0.0000      1.0
238.0            2     0.0208    0.0263      0.5
950.0            2     0.0208    0.0000      0.5
263.0            2     0.0208    0.0000      1.0
529.0            2     0.0208    0.0263      0.0

=== actualizada: milisegundos (solo filas con ms ≠ 0) ===
             filas_train  pct_train  pct_test  pct_new
actualizada                                           
636.0                 10     0.0030    0.0007   0.6000
985.0                 10     0.0030    0.0014   0.6000
20.0                   9     0.0027    0.0014   0.7778
114.0                  9     0.0027    0.0014   0.6667
417.0                  9     0.0027    0.0000   0.7778
981.0           

In [9]:
# ¿last_updated con ms ≠ 0 es solo "se actualizó más tarde" (ya está en gap_actualizacion)?
# % de new (y filas) por tramo de inicio → actualizada, con y sin ms ≠ 0. Solo train.
tramo_gap = pd.cut(dif["inicio_a_actualizada"], [-np.inf, 10, 60, 3600, DIA, 30 * DIA, np.inf],
                   labels=["≤ 10 s", "11-60 s", "1-60 min", "1-24 h", "1-30 días", "> 30 días"])
flag_ms = redondos["actualizada_ms_no_cero"].map({0: "ms = 000", 1: "ms ≠ 000"})
tr = ~es_test
print(pd.crosstab(tramo_gap[tr], flag_ms[tr], values=y.to_numpy(), aggfunc="mean").round(3))
print(pd.crosstab(tramo_gap[tr], flag_ms[tr]))

actualizada_ms_no_cero  ms = 000  ms ≠ 000
inicio_a_actualizada                      
≤ 10 s                     0.493     0.333
11-60 s                    0.268     0.464
1-60 min                   0.455     0.612
1-24 h                     0.577     0.693
1-30 días                  0.643     0.639
> 30 días                  0.670     0.714
actualizada_ms_no_cero  ms = 000  ms ≠ 000
inicio_a_actualizada                      
≤ 10 s                     39595         6
11-60 s                     1162        69
1-60 min                    6185       240
1-24 h                      1295       225
1-30 días                  13238      1835
> 30 días                   5187       963


**Lectura de la parte 3.**
- `start_time` y `date_created` **nunca** tienen milisegundos (siempre 000). `stop_time`, casi nunca (96 filas, 30% de `new`, con duraciones arbitrarias: publicaciones finalizadas a mano).
- **`last_updated` con milisegundos ≠ 000: 4,8% de las filas, 66% de `new`** (contra 53% del resto). Es la señal más visible de esta parte. Pero el control muestra que es casi toda **"se actualizó más tarde"**: el 84% de esas filas se actualizó más de un día después, y en el tramo "1-30 días" no hay diferencia (64% contra 64%). Queda una diferencia residual en "1-60 min" y "1-24 h" (~460 filas: 61-69% contra 46-58%).
- Segundos = 0 aparece en ~1,7-1,8% de las filas en las 4 fechas, lo esperable por azar (1,67%), sin diferencia de `new`. La hora en punto (hh:00:00) es rara (0,1%) y algo más `used` (20-26%), pero son ~85 filas.
- **Conclusión:** no hay una huella de carga automática en los valores redondos. El único candidato (`ms` de `last_updated`) se superpone mucho con `gap_actualizacion`.

## 4. Coincidencias exactas entre fechas

Flags de igualdad exacta (al milisegundo) entre cada par de fechas, de igualdad "a menos de 1 segundo", y dos flags de orden "raro" (actualización anterior al inicio, creación posterior al inicio). La pregunta es si alguna coincidencia separa `new` de `used`.

In [10]:
iguales = {}
for a, b in pares:
    iguales[f"{a}_igual_{b}"] = (ms[a] == ms[b]).astype(int)
    iguales[f"{a}_igual_{b}_1s"] = ((ms[a] - ms[b]).abs() <= 1000).astype(int)
iguales["actualizada_antes_de_inicio"] = (ms["actualizada"] < ms["inicio"]).astype(int)
iguales["creada_despues_de_inicio"] = (ms["creada"] > ms["inicio"]).astype(int)
iguales = pd.DataFrame(iguales)
resumen_flags(iguales)

,filas_train_1,pct_train_1,pct_test_1,pct_new_si_1,pct_new_si_0
flag,,,,,
creada_igual_inicio,55789,0.7970,0.7993,0.5352,0.5439
creada_igual_inicio_1s,69411,0.9916,0.9922,0.5380,0.4194
creada_igual_actualizada,1475,0.0211,0.0214,0.4969,0.5378
creada_igual_actualizada_1s,10432,0.1490,0.1479,0.4952,0.5443
creada_igual_fin,0,0.0000,0.0000,NaN,0.5370
creada_igual_fin_1s,0,0.0000,0.0000,NaN,0.5370
inicio_igual_actualizada,922,0.0132,0.0128,0.4414,0.5382
inicio_igual_actualizada_1s,8484,0.1212,0.1196,0.4902,0.5434
inicio_igual_fin,0,0.0000,0.0000,NaN,0.5370


**Lectura de la parte 4.**
- `date_created` = `start_time` en el 80% de las filas (y a ≤ 1 s en el 99,2%). Las que no coinciden a 1 s (0,8%) tienen 42% de `new`, contra 54%.
- `start_time` = `last_updated` exacto (1,3%): 44% de `new`. A ≤ 1 s (12%): 49%.
- `stop_time` nunca coincide con otra fecha, y `last_updated` nunca es anterior al inicio.
- **Ninguna coincidencia separa mucho**: diferencias de 5-12 puntos en grupos chicos.

## 5. Cargas masivas por vendedor

**Idea.** Una tienda que carga su catálogo por API publica decenas de artículos en pocos segundos; un particular publica de a uno. Para cada publicación, ordenando las del **mismo vendedor** por `start_time` (train + test juntos):
- `seg_desde_anterior`, `seg_hasta_siguiente`: segundos hasta la publicación anterior y la siguiente del vendedor (vacío si no hay);
- `n_mismo_segundo`, `n_misma_hora`, `n_mismo_dia`: cuántas publicó el vendedor en el mismo segundo, la misma hora y el mismo día (hora argentina), contándose a sí misma;
- `seg_vecina`: 0 si comparte el segundo con otra publicación del vendedor; si no, el mínimo entre anterior y siguiente.

"Anterior" y "siguiente" se definen sobre instantes **distintos** (ver el comentario del código): así la feature no depende del orden de las filas en los archivos.

**¿Por qué no es leakage?** Leakage es usar, al entrenar, información que no se tendría al predecir; en particular, **las etiquetas** de las filas que se evalúan. Acá:
1. Solo se usan `seller_id` y `start_time`, que están en el test. **Nunca se usa `condition`**: ni de train ni de test.
2. Se calcula sobre train + test juntos para que una fila de train y una de test se describan **igual** (sus vecinas pueden estar en cualquiera de los dos archivos). Si se calculara solo con train, las filas de test perderían a sus vecinas de train... y al revés, y la feature tendría distinta distribución en cada uno.
3. Dentro de la CV, las filas del fold de validación aportan sus **fechas** a las features de las otras filas, pero no su etiqueta: el modelo no puede "ver la respuesta". Es lo mismo que pasa con `duracion_dias`, que usa solo fechas.

**Limitación (la misma que V2 en `10_vendedor_v2`):** usa información de otras publicaciones del conjunto (*transductivo*). En la competencia es legítimo porque el test completo está disponible, pero en producción una publicación nueva no tendría "siguiente". `seg_desde_anterior` y los conteos hasta el momento sí se podrían calcular en tiempo real; `seg_hasta_siguiente` no. Hay que tenerlo en cuenta al decidir.

**Riesgo de redundancia:** "muchas publicaciones juntas" puede ser solo otra forma de decir "vendedor grande", que ya está en el modelo (`seller_id`, conteo del vendedor). Se controla abajo.

In [11]:
# Instantes DISTINTOS de cada vendedor, ordenados. Si se ordenaran las filas directamente, en un empate
# (varias publicaciones en el mismo segundo) la "anterior" dependería de la posición en el archivo (train
# antes que test), y las filas de test tendrían más "0 s" que las de train. Con instantes distintos, la
# anterior es la publicación con un start_time ESTRICTAMENTE menor: no depende del orden de las filas.
instantes = (pd.DataFrame({"vendedor": fechas["seller_id"], "t": ms["inicio"]})
             .drop_duplicates().sort_values(["vendedor", "t"]))
g = instantes.groupby("vendedor")["t"]
instantes["seg_desde_anterior"] = (instantes["t"] - g.shift(1)) / 1000
instantes["seg_hasta_siguiente"] = (g.shift(-1) - instantes["t"]) / 1000
vendedor = (pd.DataFrame({"vendedor": fechas["seller_id"], "t": ms["inicio"]})
            .merge(instantes, on=["vendedor", "t"], how="left")[["seg_desde_anterior", "seg_hasta_siguiente"]])

hora_local = (ms["inicio"] - 3 * 3600 * 1000) // (3600 * 1000)
dia_local = (ms["inicio"] - 3 * 3600 * 1000) // (DIA * 1000)
vendedor["n_mismo_segundo"] = ms["inicio"].groupby([fechas["seller_id"], ms["inicio"]]).transform("size")
vendedor["n_misma_hora"] = ms["inicio"].groupby([fechas["seller_id"], hora_local]).transform("size")
vendedor["n_mismo_dia"] = ms["inicio"].groupby([fechas["seller_id"], dia_local]).transform("size")
# Vecina más cercana: 0 si comparte el segundo con otra del vendedor; si no, la más cercana de las dos
vendedor["seg_vecina"] = vendedor[["seg_desde_anterior", "seg_hasta_siguiente"]].min(axis=1).where(vendedor["n_mismo_segundo"] == 1, 0)
assert len(vendedor) == len(fechas)
n_vendedor = fechas.groupby("seller_id")["seller_id"].transform("size")   # solo para el control de redundancia

vendedor.describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
seg_desde_anterior,62495.0,438766.6,972804.1,1.0,230.0,88094.0,508741.0,73389180.0
seg_hasta_siguiente,62581.0,433866.8,969847.1,1.0,213.0,86264.0,502527.0,73389180.0
n_mismo_segundo,100000.0,1.3,0.7,1.0,1.0,1.0,1.0,7.0
n_misma_hora,100000.0,5.0,14.3,1.0,1.0,1.0,3.0,153.0
n_mismo_dia,100000.0,6.8,18.4,1.0,1.0,1.0,4.0,153.0
seg_vecina,75022.0,356135.4,992244.2,0.0,0.0,1127.5,268752.0,73389180.0


In [12]:
cortes_seg = [-np.inf, 0, 10, 60, 600, 3600, DIA, np.inf]
etiquetas_seg = ["0 s", "1-10 s", "11-60 s", "1-10 min", "10-60 min", "1-24 h", "> 1 día"]


def tramos_seg(s):
    return pd.cut(s, cortes_seg, labels=etiquetas_seg).cat.add_categories("sin vecina").fillna("sin vecina").astype(str)


orden_seg = etiquetas_seg + ["sin vecina"]
print("=== seg_desde_anterior ===");  print(tabla(tramos_seg(vendedor["seg_desde_anterior"]), indice=orden_seg).to_string())
print("\n=== seg_vecina ===");        print(tabla(tramos_seg(vendedor["seg_vecina"]), indice=orden_seg).to_string())
tramos_n = lambda s: pd.cut(s, [0, 1, 5, 20, 100, np.inf], labels=["1", "2-5", "6-20", "21-100", "> 100"]).astype(str)
orden_n = ["1", "2-5", "6-20", "21-100", "> 100"]
print("\n=== n_mismo_segundo ===");   print(tabla(tramos_n(vendedor["n_mismo_segundo"]), indice=orden_n).to_string())
print("\n=== n_misma_hora ===");      print(tabla(tramos_n(vendedor["n_misma_hora"]), indice=orden_n).to_string())
print("\n=== n_mismo_dia ===");       print(tabla(tramos_n(vendedor["n_mismo_dia"]), indice=orden_n).to_string())

=== seg_desde_anterior ===
            filas_train  pct_train  pct_test  pct_new
0 s                   0     0.0000    0.0000      NaN
1-10 s             3278     0.0468    0.0471   0.7608
11-60 s            3554     0.0508    0.0533   0.4789
1-10 min           6236     0.0891    0.0899   0.4091
10-60 min          2700     0.0386    0.0375   0.4926
1-24 h             5820     0.0831    0.0842   0.4644
> 1 día           22171     0.3167    0.3125   0.5725
sin vecina        26241     0.3749    0.3755   0.5379

=== seg_vecina ===
            filas_train  pct_train  pct_test  pct_new
0 s               14929     0.2133    0.2176   0.4504
1-10 s             3291     0.0470    0.0484   0.7214
11-60 s            2338     0.0334    0.0331   0.5359
1-10 min           4644     0.0663    0.0657   0.4711
10-60 min          2518     0.0360    0.0351   0.5322
1-24 h             5904     0.0843    0.0811   0.5351
> 1 día           18873     0.2696    0.2699   0.6307
sin vecina        17503     0.2500 

In [13]:
# Control de redundancia: % de new por tamaño del vendedor (filas) y cercanía a su vecina (columnas), solo train
tam = pd.cut(n_vendedor, [0, 1, 5, 50, np.inf], labels=["1 publicación", "2-5", "6-50", "> 50"])
cerca = tramos_seg(vendedor["seg_vecina"])
tr = ~es_test
pd.crosstab(tam[tr], cerca[tr], values=y.to_numpy(), aggfunc="mean").reindex(columns=orden_seg).round(3)

seg_vecina,0 s,1-10 s,11-60 s,1-10 min,10-60 min,1-24 h,> 1 día,sin vecina
seller_id,,,,,,,,
1 publicación,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.494
2-5,0.602,0.606,0.636,0.563,0.577,0.640,0.672,NaN
6-50,0.473,0.609,0.600,0.502,0.570,0.582,0.591,NaN
> 50,0.374,0.798,0.438,0.357,0.389,0.323,0.354,NaN


In [14]:
pd.crosstab(tam[tr], cerca[tr]).reindex(columns=orden_seg)   # cantidad de filas de cada celda

seg_vecina,0 s,1-10 s,11-60 s,1-10 min,10-60 min,1-24 h,> 1 día,sin vecina
seller_id,,,,,,,,
1 publicación,0,0,0,0,0,0,0,17503
2-5,1886,226,423,910,565,961,10640,0
6-50,7165,1102,899,2363,1408,3657,7747,0
> 50,5878,1963,1016,1371,545,1286,486,0


**Lectura de la parte 5.**
- **Train y test tienen la misma distribución** en todas estas features (por ejemplo, "1-10 s desde la anterior": 4,68% en train y 4,71% en test). Esto confirma que calcularlas sobre los dos juntos y con instantes distintos las trata igual.
- **Publicar a 1-10 s de otra publicación del vendedor:** 76% de `new` (anterior) / 72% (vecina), contra 54% en general.
- **Más de 100 publicaciones del vendedor en la misma hora:** 536 filas, **100% `new`**. En el mismo día: 93%. Seguramente son pocas tiendas grandes, que ya identifica `seller_id`.
- **Compartir el segundo exacto** con otra publicación (`n_mismo_segundo` ≥ 2, 21% de las filas): 45% de `new`, contra 56%.
- **Control de redundancia (tabla cruzada):** entre los vendedores con más de 50 publicaciones, las que salieron a **1-10 s** de otra son **80% `new`**, contra 32-44% del resto del mismo grupo. Es una señal **dentro** del vendedor (cargas en ráfaga), no solo "vendedor grande". En los vendedores de 2-50 publicaciones la diferencia es mucho más chica.
- Es la parte con la señal más prometedora. Falta ver si el modelo ya la captura por otro lado (por ejemplo, con `seller_id` × categoría).

## 6. Medición: señal sola y aporte marginal sobre el HistGB P7a con duración

Igual que en la 18, para cada candidata:
- **(a) señal sola:** `HistGradientBoostingClassifier` con solo esa columna, CV de 5 folds;
- **(b) aporte marginal:** `armar_pipeline_histgb_combos()` de `features.py` (HistGB P7a **con** `duracion_dias`, con el modelo de texto cacheado del laboratorio) + la columna como numérica, y **comparación pareada** fold por fold contra el mismo pipeline sin ella. Pasa si gana 5/5, o 4/5 con media > desvío (diferencias redondeadas a 1e-10).

**Candidatas** (todas numéricas o flags 0/1, calculadas arriba sobre train + test):
- **A, diferencias** (segundos): las 4 que no están en el modelo (`inicio_a_actualizada` ya es `gap_actualizacion` e `inicio_a_fin` ya es `duracion_dias`) y los **sobrantes** de los 6 pares;
- **B, componentes:** los 20 menos `inicio_hora` (es `hora_inicio`, corrida 3 horas: el árbol hace los mismos cortes);
- **C, redondos:** los 10 flags que no son constantes;
- **D, coincidencias:** los 7 flags con al menos 0,1% de filas;
- **E, vendedor:** las 6 de la parte 5.

Al final, cada **familia completa** como bloque (como en `11b_marginales`: señales chicas que solas no pasan pueden pasar juntas).

Sesgo de selección: con ~57 pruebas sobre los mismos folds, alguna puede pasar "de suerte" (con 5/5 la probabilidad por prueba es ~3%, así que se esperan ~1-2 falsos positivos). Por eso lo que pase se confirma con la semilla 7 y en el promedio final.

In [15]:
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from features import (aplanar_anidadas, agregar_features_contenido, agregar_combos, armar_pipeline_histgb_combos,
                      PrediccionOOF, armar_modelo_texto)

candidatas = pd.concat([
    dif[["creada_a_inicio", "creada_a_actualizada", "creada_a_fin", "actualizada_a_fin"]].add_prefix("A_dif_"),
    sobrante.add_prefix("A_sobrante_"),
    componentes.drop(columns="inicio_hora").add_prefix("B_"),
    redondos.drop(columns=["inicio_ms_no_cero", "creada_ms_no_cero"]).add_prefix("C_"),
    iguales.loc[:, iguales[~es_test].mean() >= 0.001].add_prefix("D_"),
    vendedor.add_prefix("E_"),
], axis=1)
familias = {f: [c for c in candidatas if c.startswith(f + "_")] for f in "ABCDE"}
print({f: len(c) for f, c in familias.items()}, "| total:", candidatas.shape[1])

# Train con las mismas funciones determinísticas que el modelo final + las candidatas (solo filas de train)
train = agregar_combos(agregar_features_contenido(aplanar_anidadas(crudo)))
train = pd.concat([train, candidatas[~es_test].reset_index(drop=True)], axis=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)
archivo_log = os.path.join(carpeta_lab, "barrido_timestamps.log")


def p7a_con(columnas):
    # HistGB P7a con duración (features.py) + columnas extra como numéricas; modelo de texto cacheado
    pipeline = armar_pipeline_histgb_combos()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    if columnas:
        transformers.append(("nuevo", "passthrough", columnas))
    return pipeline


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)   # sin redondear, diferencias de 1e-17 contaban como "gana"
    gana = int((diferencias > 0).sum())
    return {"dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


# Referencia: HistGB P7a con duración, recalculada y comparada con la guardada en la 18 (oof_p7c_duracion.csv)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    inicio_ref = time.time()
    acc_ref = cross_validate(p7a_con([]), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
oof_18 = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
acc_18 = ((oof_18["histgb"] >= 0.5) == oof_18["y"]).groupby(oof_18["fold"]).mean().to_numpy()
print("Referencia recalculada:", acc_ref.round(6), "media", acc_ref.mean().round(4), f"({(time.time() - inicio_ref) / 60:.1f} min)")
print("Guardada en la 18:     ", acc_18.round(6), "| iguales:", np.allclose(acc_ref, acc_18, atol=1e-9))

{'A': 10, 'B': 19, 'C': 10, 'D': 7, 'E': 6} | total: 52


Referencia recalculada: [0.942786 0.939286 0.942571 0.939357 0.942214] media 0.9412 (0.6 min)
Guardada en la 18:      [0.942786 0.939286 0.942571 0.939357 0.942214] | iguales: True


In [16]:
def medir(nombre, columnas):
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sola = (cross_validate(HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED),
                               train[columnas], y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"].mean()
                if len(columnas) == 1 else np.nan)
        marginal = cross_validate(p7a_con(columnas), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
    fila = {"candidata": nombre, "familia": nombre[0], "señal_sola": sola, "con_P7a": marginal.mean(),
            **comparar(marginal, acc_ref), "minutos": (time.time() - inicio) / 60,
            **{f"fold_{k + 1}": a for k, a in enumerate(marginal)}}
    linea = (f"{nombre:38s} sola {sola:.4f} | con P7a {marginal.mean():.4f} | dif {fila['dif_media']:+.4f} "
             f"({fila['folds_gana']}/5){' PASA' if fila['pasa'] else ''} | {fila['minutos']:.1f} min")
    print(linea)
    with open(archivo_log, "a") as f:
        f.write(linea + "\n")
    return fila


filas = []
inicio_total = time.time()
for c in candidatas:
    filas.append(medir(c, [c]))
for f, columnas in familias.items():
    filas.append(medir(f"{f} (familia completa, {len(columnas)} columnas)", columnas))
print(f"Total: {(time.time() - inicio_total) / 60:.1f} min")

resultados = pd.DataFrame(filas).set_index("candidata")
resultados.to_csv(os.path.join(carpeta_lab, "barrido_timestamps.csv"))
resultados.drop(columns=[f"fold_{k}" for k in range(1, 6)]).sort_values("dif_media", ascending=False).round(4)

A_dif_creada_a_inicio                  sola 0.5389 | con P7a 0.9409 | dif -0.0003 (1/5) | 0.5 min


A_dif_creada_a_actualizada             sola 0.5613 | con P7a 0.9411 | dif -0.0001 (2/5) | 15.9 min


A_dif_creada_a_fin                     sola 0.5641 | con P7a 0.9412 | dif -0.0000 (2/5) | 0.6 min


A_dif_actualizada_a_fin                sola 0.5629 | con P7a 0.9411 | dif -0.0001 (2/5) | 0.6 min


A_sobrante_creada_a_inicio             sola 0.5390 | con P7a 0.9410 | dif -0.0003 (2/5) | 0.6 min


A_sobrante_creada_a_actualizada        sola 0.5611 | con P7a 0.9409 | dif -0.0003 (1/5) | 5.2 min


... (48 outputs left out)

C (familia completa, 10 columnas)      sola nan | con P7a 0.9415 | dif +0.0002 (3/5) | 0.5 min


D (familia completa, 7 columnas)       sola nan | con P7a 0.9410 | dif -0.0002 (2/5) | 0.6 min


E (familia completa, 6 columnas)       sola nan | con P7a 0.9412 | dif -0.0000 (3/5) | 0.5 min
Total: 54.3 min


,familia,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos
candidata,,,,,,,,
B_actualizada_hora,B,0.5504,0.9417,0.0005,0.0009,3,False,0.5398
A_sobrante_actualizada_a_fin,A,0.5648,0.9416,0.0003,0.0006,3,False,0.5351
E_seg_desde_anterior,E,0.5689,0.9415,0.0002,0.0008,2,False,0.5321
"C (familia completa, 10 columnas)",C,NaN,0.9415,0.0002,0.0005,3,False,0.5392
C_inicio_hora_en_punto,C,0.5376,0.9414,0.0001,0.0003,2,False,0.5245
E_n_mismo_dia,E,0.5729,0.9414,0.0001,0.0005,3,False,0.6293
B_actualizada_dia_semana,B,0.5443,0.9413,0.0001,0.0005,3,False,0.5406
B_creada_minuto,B,0.5364,0.9413,0.0001,0.0008,3,False,0.5737
E_seg_vecina,E,0.5648,0.9413,0.0001,0.0011,3,False,0.5638


## 7. Conclusión

**Ninguna candidata pasa, ni sola ni en familia.** Referencia: HistGB P7a con duración = 0,9412 (recalculada; idéntica fold por fold a la guardada en la 18).

| | Resultado |
|---|---|
| Mejores sueltas | `actualizada_hora` +0,0005 (3/5, desvío 0,0009); sobrante `actualizada_a_fin` +0,0003 (3/5); `seg_desde_anterior` +0,0002 (2/5) |
| Familia A (diferencias y sobrantes, 10) | −0,0007 (1/5) |
| Familia B (componentes, 19) | −0,0006 (2/5) |
| Familia C (ms y redondos, 10) | +0,0002 (3/5) |
| Familia D (coincidencias, 7) | −0,0002 (2/5) |
| Familia E (cargas por vendedor, 6) | −0,0000 (3/5) |
| Señal sola máxima | 0,573 (`n_mismo_dia`); todas entre 0,536 y 0,573 |

- De 57 pruebas, 15 dan media positiva y 39 negativa: es lo esperable al sumar columnas sin información nueva (cada una le da al árbol más oportunidades de cortar por ruido).
- **La señal fuerte de las fechas ya estaba entera en `duracion_dias`.** Los únicos sobrantes llamativos (+5 a +11 s, 0-2% de `new`) son esas mismas filas.
- **Cargas en ráfaga por vendedor:** la EDA mostró señal dentro de los vendedores grandes (a 1-10 s de otra publicación: 80% de `new` contra 32-44%), pero no suma sobre P7a. **Hipótesis (no verificada):** las combinaciones vendedor × categoría / tipo / precio ya separan las líneas de producto de cada tienda, y las ráfagas coinciden con esas líneas.
- **`last_updated` con milisegundos ≠ 000** (66% de `new`) es casi todo "se actualizó más tarde", que ya está en `gap_actualizacion`.

**Decisión:** no se cambia nada. Modelo final = P7c + duración (0,9420). No hace falta confirmar con la semilla 7 ni en el promedio (no hay candidatas), y no hay submission nueva. Resultados guardados en `.cache_laboratorio/barrido_timestamps.csv`.